<a href="https://colab.research.google.com/github/HelloPenguin1/Pytorch_Revision/blob/main/pytorch_prerequisite_for_transformers.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
import torch

In [2]:
import torch.nn.functional as N

In [4]:
x = torch.tensor([[4],
                 [5],
                 [6]], dtype = torch.float64)

In [5]:
x.shape

torch.Size([3, 1])

Transformer notation often uses (B, T, C) where
- B = batch size
- T = sequence length
- C = channels/embedding dimension

In [8]:
x = torch.arange(2,7, 1)
x

tensor([2, 3, 4, 5, 6])

In [16]:
x = torch.arange(12)
x


tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11])

### 💡 Why `view` / `reshape` is Critical in Transformers

While `transpose` reorders axes, `view` and `reshape` alter the structural dimensions of data. They are crucial for packing, splitting, and merging tensor shapes as data moves through the architecture.

#### 1. Splitting Multi-Head Attention
* **The Concept:** A Transformer starts with a single dense hidden state of shape `(Batch, Seq_Len, Hidden_Dim)`. To implement *Multi-Head* attention, that single `Hidden_Dim` must be sliced into multiple independent attention heads.
* **The Mechanics:** `view` splits `Hidden_Dim` into `(Heads, Head_Dim)`. The shape transitions seamlessly from `(B, S, H)` to `(B, S, Heads, Head_Dim)`.

#### 2. Flattening Heads Back Together (Concancellation)
* **The Concept:** After each attention head finishes its independent calculations, their outputs must be stitched back together into a single representation before entering the Linear/Feed-Forward layers.
* **The Mechanics:** `view` collapses the multidimensional heads back into a single vector, transforming the shape from `(B, S, Heads, Head_Dim)` back to the original unified layout: `(B, S, Hidden_Dim)`.

#### 3. Preparing Data for Feed-Forward Networks
* **The Concept:** The standard Linear (MLP) layers inside a Transformer expect data formatted for flat matrix operations.
* **The Mechanics:** Models often use `view(-1, Hidden_Dim)` to temporarily flatten the Batch and Sequence dimensions into a single long column of tokens `(Batch * Seq_Len, Hidden_Dim)`. This allows the feed-forward network to process every token simultaneously with maximum hardware efficiency.

> **⚠️ Core Difference:** `view` / `reshape` change *how* data is grouped and structured without changing the memory layout. `transpose` physically flips the axis directions.


In [18]:
#convert to matrix
x = x.view(2,6)
x

tensor([[ 0,  1,  2,  3,  4,  5],
        [ 6,  7,  8,  9, 10, 11]])

In [20]:
y = torch.arange(100)
y

tensor([ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
        18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35,
        36, 37, 38, 39, 40, 41, 42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53,
        54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67, 68, 69, 70, 71,
        72, 73, 74, 75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89,
        90, 91, 92, 93, 94, 95, 96, 97, 98, 99])

In [21]:
y.view(4,25)

tensor([[ 0,  1,  2,  3,  4,  5,  6,  7,  8,  9, 10, 11, 12, 13, 14, 15, 16, 17,
         18, 19, 20, 21, 22, 23, 24],
        [25, 26, 27, 28, 29, 30, 31, 32, 33, 34, 35, 36, 37, 38, 39, 40, 41, 42,
         43, 44, 45, 46, 47, 48, 49],
        [50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61, 62, 63, 64, 65, 66, 67,
         68, 69, 70, 71, 72, 73, 74],
        [75, 76, 77, 78, 79, 80, 81, 82, 83, 84, 85, 86, 87, 88, 89, 90, 91, 92,
         93, 94, 95, 96, 97, 98, 99]])

The fundamental difference between reshape() and view() in PyTorch is that view() strictly forces the output tensor to share the same underlying data memory as the input, while reshape() handles memory management automatically by returning a view if possible, or cloning a new copy into memory when it is not.

Use torch.Tensor.view() if you want to ensure your operations stay memory-efficient and never implicitly copy data. It acts as a safety check during deep learning pipeline design to verify that data isn't being inefficiently allocated.Use torch.Tensor.reshape() if you are writing robust scripts where logic flow safety is a higher priority than verifying low-level memory allocation quirks.

In [24]:
x = torch.arange(10)
y = x.reshape(2,5)
y

tensor([[0, 1, 2, 3, 4],
        [5, 6, 7, 8, 9]])

In [39]:
# creating a 3d tensor representing 2 images, each 3 pixel high and 4 pixels wide
# shape = (batch size, height, width) -> (2,3,4)
images = torch.arange(24).view(2,3,4)

#flatten the spatial dimensions (height and width) into a single 1d vector per image
flattened_images = images.view(2, -1)

print("Original Shape:", images.shape)            # Output: torch.Size([2, 3, 4])
print("Flattened Shape:", flattened_images.shape)  # Output: torch.Size([2, 12])


Original Shape: torch.Size([2, 3, 4])
Flattened Shape: torch.Size([2, 12])


In [41]:
#The first argument 2 tells PyTorch to keep the first dimension exactly the same. In this case, it preserves the Batch Size. We still want 2 distinct images; we just want their internal pixel grids flattened.

In [40]:
#The -1 parameter is a special wildcard/shortcut in PyTorch. It tells the computer: "Do the math for me."

In [44]:
multid = torch.arange(120).view(2,3,4,5)
multid.shape

torch.Size([2, 3, 4, 5])

In [46]:
multid.view(2,3,-1).shape

torch.Size([2, 3, 20])

In [48]:
multid.view(-1).shape

torch.Size([120])

 ## transpose()

### 💡 Why `transpose` is Critical in Self-Attention

In Transformers, `transpose` is the mathematical bridge that allows the model to compute the relationship between every token in parallel.

#### 1. Geometric Alignment (Matrix Multiplication)
* **Problem:** To compute self-attention, we must multiply the Query matrix $Q$ by the Key matrix $K$. Both share the shape `(Sequence Length, Feature Dimension)`. You cannot multiply two matrices of the exact same orientation.
* **Solution:** Transposing $K$ flips it sideways into $K^T$, changing its shape to `(Feature Dimension, Sequence Length)`.

#### 2. Generating the Attention Map
* **The Math:** $Attention\_Scores = QK^T$
* **The Result:** Multiplying `(Seq_Len, Features)` by `(Features, Seq_Len)` yields a square matrix of shape `(Seq_Len, Seq_Len)`.
* **The Concept:** This resulting grid maps every word in the sequence against every other word, creating the "Attention Map" that allows tokens to contextually weight each other.

#### 3. Multi-Head Isolation
* Multi-head attention formats data as a 4D tensor: `(Batch, Seq_Len, Heads, Head_Dim)`.
* We use `transpose(1, 2)` to swap `Seq_Len` and `Heads`. This groups all tokens together per head so matrix multiplication can happen cleanly and independently across parallel attention heads.


transpose can only work on pairs


In [26]:
#swaps TWO dimensions
x = torch.Tensor([
    [1,2,3],
    [4,5,6]
])
x.transpose(0,1)

tensor([[1., 4.],
        [2., 5.],
        [3., 6.]])

In [30]:
images = torch.rand(32, 3, 224, 224)
images.shape

torch.Size([32, 3, 224, 224])

In [35]:
images.shape[-1]

224

In [37]:
flipped_image = images.transpose(1,2)
flipped_image.shape

torch.Size([32, 224, 3, 224])

Permute: reorders MANY dimnsiosn

In [52]:
x = torch.randn(2,3,4)
x.shape

torch.Size([2, 3, 4])

In [53]:
y = x.permute(2,0,1)
y.shape

torch.Size([4, 2, 3])

In [54]:
y = x.permute(2,1,0)
y.shape

torch.Size([4, 3, 2])

In [58]:
x = torch.tensor([[2,3,4,5],
                  [6,7,8,9],
                  [10,11,12,13]])
x.shape

torch.Size([3, 4])

In [59]:
y = x.permute(1,0)
y

tensor([[ 2,  6, 10],
        [ 3,  7, 11],
        [ 4,  8, 12],
        [ 5,  9, 13]])

- Use .transpose() when you want to flip adjacent matrix dimensions—specifically the last two dimensions during a math operation (like \(QK^{T}\)). Writing k.transpose(-2, -1) reads clearly as "matrix transpose the end features.

- Use .permute() when you are changing the entire architectural layout of a data block (e.g., converting an entire vision tensor from Channels-First (B, C, H, W) to Channels-Last (B, H, W, C)).

Unsqueeze

In [61]:
#adds dimension of size 1
x = torch.tensor([1,2,3,4])
x.shape

torch.Size([4])

In [63]:
y =x.unsqueeze(0)
y.shape

torch.Size([1, 4])

In [64]:
z = y.unsqueeze(-1) # add dim at last position
z.shape

torch.Size([1, 4, 1])

Squueze

In [81]:
#removes dimensions of size 1
x = torch.rand(1,2,3,4)
x.shape

torch.Size([1, 2, 3, 4])

In [84]:
x = x.squeeze()

In [85]:
x.shape

torch.Size([2, 3, 4])

# matmul (@): matrix multiplication

we find the attention score using this

In [86]:
x = torch.tensor([
    [1,2],
    [3,4]
])

y = torch.tensor([
    [2,3,4,5],
    [6,7,8,8]
])

z = x @ y
print(z)

tensor([[14, 17, 20, 21],
        [30, 37, 44, 47]])


#Masked_fill()

In [89]:
#replaces selected values
#used for causal masking in GPT

x = torch.tensor([
    [1.,2.,3.],
    [4.,5.,6.]
])

mask = torch.tensor([
    [True, False, True],
    [False, True, False]
])

y = x.masked_fill(mask, -999)

print(y)

tensor([[-999.,    2., -999.],
        [   4., -999.,    6.]])


# Softmax: to convert score into probabilities

In [91]:
x = torch.tensor([1,2,3,99])
y = N.softmax(x.float(), dim=0)
print(y)

tensor([2.7465e-43, 7.4689e-43, 2.0305e-42, 1.0000e+00])


In [92]:
y.sum()

tensor(1.)

# cat():  concats exisiting dims

In [100]:
a = torch.tensor([
    [1,2],
    [3,4]
])

b = torch.tensor([
    [5,6],
    [7,8]
])

y = torch.cat([a,b], dim=0)

print(y)
print(y.shape)

tensor([[1, 2],
        [3, 4],
        [5, 6],
        [7, 8]])
torch.Size([4, 2])


concatate vertcically


In [101]:
z = torch.cat([a,b], dim=1)
z


tensor([[1, 2, 5, 6],
        [3, 4, 7, 8]])

# stack(): creates new dimension

In [96]:
a = torch.tensor([1,2])
b = torch.tensor([3,4])

y = torch.stack([a,b])

print(y)

tensor([[1, 2],
        [3, 4]])
